# Proyecto ETL – Contaminación, Mortalidad y Población por Localidad (Bogotá)

Este es parte de nuestro ETL parcial

- **IBOCA** (calidad del aire por estación)
- **SISAIRE** (contaminantes por estación/localidad)
- **Medicina** (mortalidad por localidad)
- **Población** (población por localidad, año y sexo)

El objetivo final es generar **6 tablas** limpias y consistentes:

1. `dim_fecha`
2. `dim_localidad`
3. `dim_localidad_fecha`  
4. `fact_mortalidad`
5. `fact_momento`
6. `fact_tasas_mortalidad` (Localidad + Fecha con población total, hombres y mujeres)

En las siguientes celdas haremos:

0. Setup general (imports, rutas).
1. Carga de datos "raw".
2. Limpieza y normalización por fuente.
3. Agregación a nivel **mes–localidad**.
4. Construcción de dimensiones.
5. Construcción de tablas de hechos.
6. Export final de CSV.


In [50]:
# 0. SETUP GENERAL

import pandas as pd
from pathlib import Path
import re
import unicodedata
import difflib

# ---------------------------- RUTAS ---------------------------------

RUTA_SISAIRE   = Path("Datos/SISAIRE")
RUTA_IBOCA     = Path("Datos/IBOCA")
RUTA_MEDICINA  = Path("Datos/Medicina")
RUTA_POBLACION = Path("Datos/datospoblacion.csv")
RUTA_SALIDA    = Path("Output_ETL")

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)


# --------------------- NOMBRES DE MESES ------------------------------

MAPA_NOMBRE_MES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}


# =====================================================================
#              0.1 LISTA MANUAL DE LOCALIDADES CANÓNICAS
# =====================================================================

MANUAL_CANONICAL_LOCALIDADES = [
    "Bogota",
    "Usaquen",
    "Chapinero",
    "Santa Fe",
    "San Cristobal",
    "Usme",
    "Tunjuelito",
    "Bosa",
    "Kennedy",
    "Fontibon",
    "Engativa",
    "Suba",
    "Barrios Unidos",
    "Teusaquillo",
    "Los Martires",
    "Antonio Narino",
    "Puente Aranda",
    "La Candelaria",
    "Rafael Uribe Uribe",
    "Ciudad Bolivar",
    "Bolivia",        # Caso especial: NO es Ciudad Bolivar
    "Sumapaz",
]


# =====================================================================
#          0.2 Helper: limpieza fuerte para localidades base
# =====================================================================

def _preprocesar_localidad_base(x: str) -> str:
    """
    Normaliza nombres de localidad **solo para generar la lista canónica**, sin fuzzy.
    Corrige encoding, quita símbolos, números y estandariza.
    """
    if not isinstance(x, str):
        return ""

    s = x.strip()

    # Intentar reparar errores típicos de encoding (Usaqua©n → Usaquén)
    try:
        s2 = s.encode("latin-1").decode("utf-8")
        s = s2
    except Exception:
        pass

    # Quitar BOM raro
    s = s.replace("\ufeff", "").lower().strip()

    # Quitar tildes
    s_norm = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s_norm if not unicodedata.combining(c))

    # Dejar solo letras y espacios (borrar ¡ ± 3 ® etc.)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if not s:
        return ""

    return s.title()



# =====================================================================
#      0.3 Cargar rápido población PARA DEFINIR LOCALIDADES CANÓNICAS
# =====================================================================

localidades_pob = []

try:
    raw_poblacion_fast = pd.read_csv(RUTA_POBLACION, sep=";", encoding="latin-1")

    if "NOMBRE_LOCALIDAD" in raw_poblacion_fast.columns:
        localidades_pob = (
            raw_poblacion_fast["NOMBRE_LOCALIDAD"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )
        print(f"Se detectaron {len(localidades_pob)} localidades distintas en datos de población.")
    else:
        print("⚠️ 'NOMBRE_LOCALIDAD' no está en el CSV de población.")
except Exception as e:
    print(f"⚠️ No se pudo cargar población para extraer localidades: {e}")



# =====================================================================
#      0.4 Construir lista CANÓNICA = manual + las de población limpia
# =====================================================================

canon_from_pob = {_preprocesar_localidad_base(loc) for loc in localidades_pob}
canon_from_pob = {c for c in canon_from_pob if c}   # Quitar vacíos

# Unir manual + detectadas automáticamente
CANONICAL_LOCALIDADES = sorted(set(MANUAL_CANONICAL_LOCALIDADES) | canon_from_pob)

# Asegurar que Bolivia esté
if "Bolivia" not in CANONICAL_LOCALIDADES:
    CANONICAL_LOCALIDADES.append("Bolivia")

CANONICAL_LOCALIDADES_L = [c.lower() for c in CANONICAL_LOCALIDADES]

print("\nLocalidades canónicas usadas para fuzzy matching:")
print(CANONICAL_LOCALIDADES)
print("\nCantidad total:", len(CANONICAL_LOCALIDADES), "\n")



# =====================================================================
#                   0.5 FUNCIÓN FINAL DE NORMALIZACIÓN
# =====================================================================

def normalizar_localidad_texto(x):
    """
    Normaliza nombres de localidad para TODAS las bases de datos:
    - Corrige encoding
    - Quita basura semántica ("localidad", "bogotá", "ciudad", etc.)
    - Fuzzy matching contra CANONICAL_LOCALIDADES
    - Caso especial: 'Bolivia' ≠ 'Ciudad Bolivar'
    """
    if not isinstance(x, str):
        return x

    s_original = x
    s = x.strip()

    # Reparar errores comunes de encoding
    try:
        s = s.encode("latin-1").decode("utf-8")
    except Exception:
        pass

    s = s.replace("\ufeff", "").lower().strip()

    # Quitar prefijos tipo "01 - "
    s = re.sub(r"^\d+\s*-\s*", "", s)

    # Quitar palabras irrelevantes
    basura = ["localidad", "loc.", "ciudad", "ciudad de", "bogota", "bogotá", "d.c."]
    for w in basura:
        s = s.replace(w, "")

    s = re.sub(r"\s+", " ", s).strip()

    # Quitar tildes
    s_norm = unicodedata.normalize("NFKD", s)
    s_sin = "".join(c for c in s_norm if not unicodedata.combining(c))

    # Caso especial Bolivia
    if re.search(r"\bbolivia\b", s_sin):
        return "Bolivia"

    # Fuzzy
    if s_sin:
        matches = difflib.get_close_matches(
            s_sin,
            CANONICAL_LOCALIDADES_L,
            n=1,
            cutoff=0.70
        )
        if matches:
            idx = CANONICAL_LOCALIDADES_L.index(matches[0])
            return CANONICAL_LOCALIDADES[idx]

    # Fallback: devolver capitalizado
    if s_sin:
        return s_sin.title()

    return s_original.strip()


print("Rutas configuradas correctamente. Carpeta de salida:", RUTA_SALIDA)


Se detectaron 21 localidades distintas en datos de población.

Localidades canónicas usadas para fuzzy matching:
['Antonio Narino', 'Barrios Unidos', 'Bogota', 'Bolivia', 'Bosa', 'Chapinero', 'Ciudad Bolivar', 'Engativa', 'Fontibon', 'Kennedy', 'La Candelaria', 'Los Martires', 'Puente Aranda', 'Rafael Uribe Uribe', 'San Cristobal', 'Santa Fe', 'Suba', 'Sumapaz', 'Teusaquillo', 'Tunjuelito', 'Usaquen', 'Usme']

Cantidad total: 22 

Rutas configuradas correctamente. Carpeta de salida: Output_ETL


## 1. Carga de datos "raw"

En esta sección solo **leemos** los archivos y los concatenamos, sin hacer limpiezas profundas.

- `raw_iboca`  ← todos los archivos de IBOCA.
- `raw_sisaire` ← todos los archivos de SISAIRE.
- `raw_medicina` ← el archivo de mortalidad.
- `raw_poblacion` ← el archivo de poblacion.

Más adelante vamos a limpiarlos y normalizarlos.


In [51]:
# 1.1 CARGA IBOCA → raw_iboca

import os
import numpy as np

def to_num(s):
    """
    Limpia valores numéricos de IBOCA:
    - NaN si viene vacío o como 'sin data'
    - Reemplaza coma por punto
    - Intenta convertir a float
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s.lower().startswith("sin data"):
        return np.nan
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan


def cargar_iboca(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Recorre todos los archivos de IBOCA en la carpeta y arma un DataFrame unificado,
    usando el formato real observado:
      - Columna 0: FechaHora (desde fila 7 en adelante)
      - Fila 5 (índice 4): nombre de la estación para cada bloque de 3 columnas
      - Cada bloque: [Concentracion, NowCast, IBOCA]
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".xlsx", ".xls", ".csv"]:
            continue
        if fichero.name.startswith("~$"):  # evitar archivos temporales de Excel
            continue

        print("Leyendo IBOCA:", fichero.name)

        # Leer archivo crudo
        try:
            if fichero.suffix.lower() in [".xlsx", ".xls"]:
                df_raw = pd.read_excel(fichero, header=None)
            else:
                df_raw = pd.read_csv(fichero, header=None)
        except Exception as e:
            print(f"  [ERROR] No se pudo leer {fichero.name}: {e}")
            continue

        # Chequeo mínimo de formato
        if df_raw.shape[0] <= 7 or df_raw.shape[1] < 4:
            print(f"  [ADVERTENCIA] Formato inesperado en {fichero.name}, se omite.")
            continue

        # Columna de fechas/horas (col 0, desde fila 7 en adelante)
        df_fechas = df_raw.iloc[7:, [0]].copy()
        df_fechas.columns = ["FechaHora"]
        df_fechas["FechaHora"] = pd.to_datetime(
            df_fechas["FechaHora"],
            errors="coerce",
            dayfirst=True
        )

        n_cols = df_raw.shape[1]

        # Recorrer bloques de 3 columnas: [Concentracion, NowCast, IBOCA]
        for col_inicio in range(1, n_cols, 3):
            if col_inicio + 2 >= n_cols:
                break

            # Nombre de la estación en la fila 5 (índice 4) de la primera columna del bloque
            nombre_estacion = df_raw.iat[4, col_inicio] if 4 < df_raw.shape[0] else None
            if not isinstance(nombre_estacion, str):
                continue
            nombre_estacion = nombre_estacion.strip()
            if not nombre_estacion or nombre_estacion.lower() == "nan":
                continue

            # Valores del bloque (desde fila 7 hacia abajo)
            vals = df_raw.iloc[7:, col_inicio:col_inicio+3].copy()
            vals.columns = ["Concentracion", "NowCast", "IBOCA"]

            # Limpiar valores numéricos
            for c in ["Concentracion", "NowCast", "IBOCA"]:
                vals[c] = vals[c].apply(to_num)

            # Unir fechas + valores
            df_block = pd.concat(
                [df_fechas.reset_index(drop=True),
                 vals.reset_index(drop=True)],
                axis=1
            )
            df_block["Location"] = nombre_estacion

            # Eliminar filas donde todo el bloque numérico está vacío
            df_block = df_block.dropna(
                subset=["Concentracion", "NowCast", "IBOCA"],
                how="all"
            )

            if not df_block.empty:
                dfs.append(df_block)

    if not dfs:
        print("No se cargó ningún bloque de IBOCA. Revisa el formato o la ruta.")
        return pd.DataFrame(
            columns=["FechaHora", "Location", "Concentracion", "NowCast", "IBOCA"]
        )

    df_iboca = pd.concat(dfs, ignore_index=True)

    # Ordenar columnas de forma coherente
    df_iboca = df_iboca[["FechaHora", "Location", "Concentracion", "NowCast", "IBOCA"]]

    print("raw_iboca cargado. Shape:", df_iboca.shape)
    return df_iboca


raw_iboca = cargar_iboca(RUTA_IBOCA)
raw_iboca.head()


Leyendo IBOCA: IBOCA-PM10-2021-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-3.xlsx
Leyendo IBOCA: IBOCA-PM10-2024-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2021-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2024-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2022-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2022-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2020-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2025-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2020-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2025-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2020-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2025-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2020-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2025-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2022-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2022-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-2.xlsx
Leyendo IBOCA: IBOCA-PM25-2023-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2021-2.xlsx
Leyendo IBOCA: IBOCA-PM10-2024-1.xlsx
Leyendo IBOCA: IBOCA-PM10-2023-3.xlsx
Leyendo IBOCA: IBOCA-PM25-2021-1.xlsx
Leyendo IBOCA: IBOCA-PM25-2024-2.xlsx
raw_iboca ca

,FechaHora,Location,Concentracion,NowCast,IBOCA
0,2021-01-01 00:00:00,Bolivia,32.0,31.0,90.58
1,2021-01-01 01:00:00,Bolivia,37.0,31.4,91.43
2,2021-01-01 02:00:00,Bolivia,62.0,33.1,95.07
3,2021-01-01 03:00:00,Bolivia,62.0,34.7,98.50
4,2021-01-01 04:00:00,Bolivia,94.0,37.9,106.12


In [52]:
# 1.2 CARGA SISAIRE → raw_sisaire

def cargar_sisaire(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Lee todos los archivos de SISAIRE (csv/xlsx) y los concatena.
    Asume que todos tienen columnas compatibles.
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".xlsx", ".xls", ".csv"]:
            continue

        print("Leyendo SISAIRE:", fichero.name)

        if fichero.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(fichero)
        else:
            df = pd.read_csv(fichero)

        dfs.append(df)

    if not dfs:
        print("No se cargó ningún archivo de SISAIRE.")
        return pd.DataFrame()

    df_sisaire = pd.concat(dfs, ignore_index=True)
    print("raw_sisaire cargado. Shape:", df_sisaire.shape)
    return df_sisaire


raw_sisaire = cargar_sisaire(RUTA_SISAIRE)
raw_sisaire.head()


Leyendo SISAIRE: SISAIRE-NO2-2025.csv
Leyendo SISAIRE: SISAIRE-NO2-2024.csv
Leyendo SISAIRE: SISAIRE-O3-2024.csv
Leyendo SISAIRE: SISAIRE-O3-2025.csv
Leyendo SISAIRE: SISAIRE-O3-2021.csv
Leyendo SISAIRE: SISAIRE-NO2-2023.csv
Leyendo SISAIRE: SISAIRE-NO2-2022.csv
Leyendo SISAIRE: SISAIRE-O3-2020.csv
Leyendo SISAIRE: SISAIRE-O3-2022.csv
Leyendo SISAIRE: SISAIRE-NO2-2020.csv
Leyendo SISAIRE: SISAIRE-NO2-2021.csv
Leyendo SISAIRE: SISAIRE-O3-2023.csv
Leyendo SISAIRE: SISAIRE-SO2-2023.csv
Leyendo SISAIRE: SISAIRE-CO-2023.csv
Leyendo SISAIRE: SISAIRE-CO-2022.csv
Leyendo SISAIRE: SISAIRE-SO2-2022.csv
Leyendo SISAIRE: SISAIRE-SO2-2020.csv
Leyendo SISAIRE: SISAIRE-CO-2020.csv
Leyendo SISAIRE: SISAIRE-CO-2021.csv
Leyendo SISAIRE: SISAIRE-SO2-2021.csv
Leyendo SISAIRE: SISAIRE-SO2-2025.csv
Leyendo SISAIRE: SISAIRE-CO-2025.csv
Leyendo SISAIRE: SISAIRE-CO-2024.csv
Leyendo SISAIRE: SISAIRE-SO2-2024.csv
raw_sisaire cargado. Shape: (2622273, 7)


,Estacion,Fecha inicial,Fecha final,NO2,O3,SO2,CO
0,"""USME""",2025-07-31 22:00,2025-07-31 22:59,39.30036,NaN,NaN,NaN
1,"""USME""",2025-07-31 21:00,2025-07-31 21:59,44.94156,NaN,NaN,NaN
2,"""USME""",2025-07-31 20:00,2025-07-31 20:59,44.00136,NaN,NaN,NaN
3,"""USME""",2025-07-31 19:00,2025-07-31 19:59,27.07776,NaN,NaN,NaN
4,"""USME""",2025-07-31 18:00,2025-07-31 18:59,11.09436,NaN,NaN,NaN


In [53]:
# 1.3 CARGA MEDICINA → raw_medicina (versión con encoding)

def cargar_medicina(ruta_carpeta: Path) -> pd.DataFrame:
    """
    Carga el/los archivos de mortalidad (Medicina).
    Intenta varios encodings típicos en datos con tildes/ñ.
    """
    dfs = []

    for fichero in ruta_carpeta.iterdir():
        if not fichero.suffix.lower() in [".csv", ".xlsx", ".xls"]:
            continue

        print("Leyendo Medicina:", fichero.name)

        if fichero.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(fichero)
        else:
            # Intentamos primero con ; y latin-1
            try:
                df = pd.read_csv(fichero, sep=";", encoding="latin-1")
            except UnicodeDecodeError:
                # Plan B: ISO-8859-1
                try:
                    df = pd.read_csv(fichero, sep=";", encoding="ISO-8859-1")
                except UnicodeDecodeError:
                    # Último recurso: sin separador explícito pero con latin-1
                    df = pd.read_csv(fichero, encoding="latin-1")

        dfs.append(df)

    if not dfs:
        print("No se cargó ningún archivo de Medicina.")
        return pd.DataFrame()

    df_med = pd.concat(dfs, ignore_index=True)
    print("raw_medicina cargado. Shape:", df_med.shape)
    return df_med

raw_medicina = cargar_medicina(RUTA_MEDICINA)
raw_medicina.head()


Leyendo Medicina: datosmedicina.csv
raw_medicina cargado. Shape: (47542, 12)


,ANO,MES,EPS,SUBRED,SEXO,MIGRANTE,REGIMEN_SEGURIDAD_SOCIAL,EDAD_FALLECIDO,EDAD_QUINQUENAL,CIE10_AGRUPADA,CIE10_BASICA,LOCALIDAD
0,2015,2,ALIANSALUD E.P.S.,NORTE,FEMENINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
1,2015,4,OTROS,NORTE,FEMENINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
2,2021,1,E.P.S. SANITAS,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
3,2021,6,E.P.S. SANITAS,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá
4,2022,2,FAMISANAR E.P.S. LTDA - CAFAM - COLSUBSIDIO,NORTE,MASCULINO,Colombiano,CONTRIBUTIVO,66,65 a 69 años,ENFERMEDADES CARDIOCEREBROVASCULARES,I219,00 - Bogotá


In [54]:
# 1.4 CARGA POBLACIÓN → raw_poblacion (versión corregida)

# 1) Leer el archivo usando latin-1 porque los acentos vienen en ese encoding.
raw_poblacion = pd.read_csv(
    RUTA_POBLACION,
    sep=";", 
    encoding="latin-1"
)

# 2) Quitar el BOM de la primera columna si existe
raw_poblacion.columns = [
    col.replace("ï»¿", "") for col in raw_poblacion.columns
]

# 3) Arreglar textos mal decodificados: UsaquÃ©n → Usaquén
def fix_text(x):
    if isinstance(x, str):
        try:
            return x.encode("latin-1").decode("utf-8")
        except:
            return x
    return x

raw_poblacion = raw_poblacion.applymap(fix_text)

print("raw_poblacion cargado. Shape:", raw_poblacion.shape)
raw_poblacion.head()


/var/folders/jq/wqkcvg155zg7f2xm3sy80x5h0000gn/T/ipykernel_17878/4147251056.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  raw_poblacion = raw_poblacion.applymap(fix_text)


raw_poblacion cargado. Shape: (131502, 10)


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,ORDEN_MCV,CURSODEVIDA,ORDEN_GRUPO_EDAD,GRUPOEDAD,POBLACION_7
0,2005,1,Usaquén,Hombres,0,1,Primera Infancia,1,0 a 4,2909
1,2005,1,Usaquén,Hombres,1,1,Primera Infancia,1,0 a 4,2954
2,2005,1,Usaquén,Hombres,2,1,Primera Infancia,1,0 a 4,2919
3,2005,1,Usaquén,Hombres,3,1,Primera Infancia,1,0 a 4,2989
4,2005,1,Usaquén,Hombres,4,1,Primera Infancia,1,0 a 4,3079


## 2. Limpieza y normalización

En esta sección transformamos cada fuente a una versión **clean** con las columnas
que vamos a usar después:

- `iboca_clean`   → FechaHora, Año, Mes, Localidad, Concentración, NowCast, IBOCA.
- `sisaire_clean` → Fecha, Año, Mes, Localidad, NO2, O3, SO2, CO.
- `medicina_clean`→ Año, Mes, Localidad, Fecha (primer día del mes), métricas de muertes.
- `poblacion_clean` → Año, Código_Localidad, Localidad, Poblacion_Total, Hombres, Mujeres.


In [55]:
# 2.1 LIMPIEZA IBOCA → iboca_clean (con fuzzy matching directo)

def limpiar_iboca(raw_iboca: pd.DataFrame) -> pd.DataFrame:
    df = raw_iboca.copy()

    # --- Asegurar tipo datetime ---
    df["FechaHora"] = pd.to_datetime(df["FechaHora"], errors="coerce")

    # --- Año y Mes ---
    df["Ano"] = df["FechaHora"].dt.year
    df["Mes"] = df["FechaHora"].dt.month

    # --- Asegurar numéricos (ya vienen limpios, reforzamos) ---
    for col in ["Concentracion", "NowCast", "IBOCA"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Localidad por FUZZY MATCHING, sin diccionario manual ---
    df["Localidad"] = df["Location"].apply(normalizar_localidad_texto)

    # --- Selección y orden de columnas finales ---
    columnas_finales = [
        "FechaHora", "Ano", "Mes", "Localidad",
        "Concentracion", "NowCast", "IBOCA"
    ]

    df = df[columnas_finales].dropna(subset=["FechaHora"])

    print("iboca_clean listo. Shape:", df.shape)
    return df


iboca_clean = limpiar_iboca(raw_iboca)
iboca_clean.head()


iboca_clean listo. Shape: (1640005, 7)


,FechaHora,Ano,Mes,Localidad,Concentracion,NowCast,IBOCA
0,2021-01-01 00:00:00,2021.0,1.0,Bolivia,32.0,31.0,90.58
1,2021-01-01 01:00:00,2021.0,1.0,Bolivia,37.0,31.4,91.43
2,2021-01-01 02:00:00,2021.0,1.0,Bolivia,62.0,33.1,95.07
3,2021-01-01 03:00:00,2021.0,1.0,Bolivia,62.0,34.7,98.50
4,2021-01-01 04:00:00,2021.0,1.0,Bolivia,94.0,37.9,106.12


In [56]:
# 2.2 LIMPIEZA SISAIRE → sisaire_clean (usando 'Fecha final' y 'Estacion')

def limpiar_sisaire(raw_sisaire: pd.DataFrame) -> pd.DataFrame:
    df = raw_sisaire.copy()

    print("Columnas originales de SISAIRE:")
    print(df.columns.tolist())

    # 1) Renombrar columnas clave a nombres estándar
    df = df.rename(
        columns={
            "Fecha final": "Fecha",
            "Estacion": "Localidad",
        }
    )

    # 2) Convertir Fecha a datetime
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

    # 3) Año y mes
    df["Ano"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.month

    # 4) Normalizar texto de localidad (AQUÍ entra TODO el fuzzy matching + caso Bolivia)
    df["Localidad"] = df["Localidad"].apply(normalizar_localidad_texto)

    # 5) Limpiar contaminantes numéricos
    contaminantes = ["NO2", "O3", "SO2", "CO"]

    for col in contaminantes:
        if col in df.columns:
            # puedes usar el mismo patrón que en IBOCA
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .str.replace("sin dato", "", case=False, regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 6) Columnas finales ordenadas
    cols_finales = ["Fecha", "Ano", "Mes", "Localidad"] + [
        c for c in contaminantes if c in df.columns
    ]
    df = df[cols_finales].dropna(subset=["Fecha"])

    print("sisaire_clean listo. Shape:", df.shape)
    return df


sisaire_clean = limpiar_sisaire(raw_sisaire)
sisaire_clean.head()


Columnas originales de SISAIRE:
['Estacion', 'Fecha inicial', 'Fecha final', 'NO2', 'O3', 'SO2', 'CO']
sisaire_clean listo. Shape: (2622273, 8)


,Fecha,Ano,Mes,Localidad,NO2,O3,SO2,CO
0,2025-07-31 22:59:00,2025,7,Usme,39.30036,NaN,NaN,NaN
1,2025-07-31 21:59:00,2025,7,Usme,44.94156,NaN,NaN,NaN
2,2025-07-31 20:59:00,2025,7,Usme,44.00136,NaN,NaN,NaN
3,2025-07-31 19:59:00,2025,7,Usme,27.07776,NaN,NaN,NaN
4,2025-07-31 18:59:00,2025,7,Usme,11.09436,NaN,NaN,NaN


In [57]:
# 2.3 LIMPIEZA MEDICINA → medicina_clean (con fuzzy matching de Localidad)

def limpiar_medicina(raw_medicina: pd.DataFrame) -> pd.DataFrame:
    df = raw_medicina.copy()

    # --- 1. Renombrar columnas estándar ---
    df.rename(columns={
        "ANO": "Ano",
        "MES": "Mes",
        "LOCALIDAD": "Localidad",
        "SEXO": "Sexo",
        "CIE10_AGRUPADA": "CIE10_Agrupada",
        "CIE10_BASICA": "CIE10_Basica",
        "REGIMEN_SEGURIDAD_SOCIAL": "Regimen",
    }, inplace=True)

    # Asegurar que Año y Mes sean numéricos
    df["Ano"] = pd.to_numeric(df["Ano"], errors="coerce")
    df["Mes"] = pd.to_numeric(df["Mes"], errors="coerce")

    # --- 2. Normalizar localidad (AQUÍ entra fuzzy + caso Bolivia) ---
    df["Localidad"] = df["Localidad"].apply(normalizar_localidad_texto)

    # --- 3. Crear fecha YYYY-MM-01 ---
    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    # --- 4. Crear columnas de conteo ---
    # Cada fila = 1 muerte
    df["Muertes_Totales"] = 1

    # Por sexo
    sexo = df["Sexo"].astype(str).str.strip().str.upper()
    df["Muertes_Totales_Hombres"] = (sexo.str.startswith("M")).astype(int)
    df["Muertes_Totales_Mujeres"] = (sexo.str.startswith("F")).astype(int)

    # Por CIE10 agrupada y básica (cada fila cuenta como 1)
    df["Muertes_CIE10_Agrupada"] = 1
    df["Muertes_CIE10_Basica"] = 1

    # --- 5. Agrupar por año, mes, localidad ---
    agg_cols = {
        "Muertes_Totales": "sum",
        "Muertes_Totales_Hombres": "sum",
        "Muertes_Totales_Mujeres": "sum",
        "Muertes_CIE10_Agrupada": "sum",
        "Muertes_CIE10_Basica": "sum",
    }

    df_group = (
        df
        .groupby(["Ano", "Mes", "Localidad", "Fecha"], as_index=False)
        .agg(agg_cols)
    )

    print("medicina_clean lista. Shape:", df_group.shape)
    return df_group


# Ejecutar
medicina_clean = limpiar_medicina(raw_medicina)
medicina_clean.head()


medicina_clean lista. Shape: (2482, 9)


,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,2015,1,00 - Bogotá,2015-01-01,188,112,76,188,188
1,2015,1,Antonio Narino,2015-01-01,2,1,1,2,2
2,2015,1,Barrios Unidos,2015-01-01,6,3,3,6,6
3,2015,1,Bolivia,2015-01-01,16,7,9,16,16
4,2015,1,Bosa,2015-01-01,20,14,6,20,20


In [58]:
# 2.4 LIMPIEZA POBLACIÓN → poblacion_clean (normalizando Localidad con fuzzy)

def limpiar_poblacion(raw_poblacion: pd.DataFrame) -> pd.DataFrame:
    df = raw_poblacion.copy()

    # --- 1. Asegurarnos de que POBLACION_7 sea numérico ---
    df["POBLACION_7"] = pd.to_numeric(df["POBLACION_7"], errors="coerce").fillna(0)

    # --- 2. Agregar por año, localidad, sexo ---
    df_agg = (
        df
        .groupby(["ANO", "CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD", "SEXO"], as_index=False)
        .agg({"POBLACION_7": "sum"})
    )

    # --- 3. Pivotear para tener columnas de hombres/mujeres ---
    df_pivot = df_agg.pivot_table(
        index=["ANO", "CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD"],
        columns="SEXO",
        values="POBLACION_7",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    df_pivot.columns.name = None  # quitar el nombre del índice de columnas

    # Detectar nombres de columnas de sexo
    col_hombres = "Hombres" if "Hombres" in df_pivot.columns else "HOMBRES"
    col_mujeres = "Mujeres" if "Mujeres" in df_pivot.columns else "MUJERES"

    df_pivot["Poblacion_Hombres"] = df_pivot.get(col_hombres, 0)
    df_pivot["Poblacion_Mujeres"] = df_pivot.get(col_mujeres, 0)
    df_pivot["Poblacion_Total"] = (
        df_pivot["Poblacion_Hombres"] + df_pivot["Poblacion_Mujeres"]
    )

    # --- 4. Renombrar columnas estándar ---
    df_pivot.rename(
        columns={
            "ANO": "Ano",
            "CODIGO_LOCALIDAD": "Codigo_Localidad",
            "NOMBRE_LOCALIDAD": "Localidad",
        },
        inplace=True,
    )

    # Asegurar tipos básicos
    df_pivot["Ano"] = pd.to_numeric(df_pivot["Ano"], errors="coerce")
    df_pivot["Codigo_Localidad"] = pd.to_numeric(df_pivot["Codigo_Localidad"], errors="coerce")

    # --- 5. Normalizar texto de localidad con fuzzy ---
    # (normalizar_localidad_texto ya tiene fallback al original si queda vacío)
    df_pivot["Localidad"] = df_pivot["Localidad"].apply(normalizar_localidad_texto)

    # --- 6. Columnas finales ---
    columnas_finales = [
        "Ano", "Codigo_Localidad", "Localidad",
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
    ]
    df_final = df_pivot[columnas_finales].copy()

    print("poblacion_clean lista. Shape:", df_final.shape)
    print("Ejemplo de localidades en poblacion_clean:", df_final["Localidad"].unique()[:10])
    return df_final


# Ejecutar limpieza
poblacion_clean = limpiar_poblacion(raw_poblacion)
poblacion_clean.head()


poblacion_clean lista. Shape: (651, 6)
Ejemplo de localidades en poblacion_clean: ['Bogotá' 'Usaquen' 'Chapinero' 'Santa Fe' 'San Cristobal' 'Usme'
 'Tunjuelito' 'Bosa' 'Kennedy' 'Fontibon']


,Ano,Codigo_Localidad,Localidad,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres
0,2005,0,Bogotá,6710910,3236477,3474433
1,2005,1,Usaquen,415099,191068,224031
2,2005,2,Chapinero,121307,55515,65792
3,2005,3,Santa Fe,96046,48127,47919
4,2005,4,San Cristobal,400187,195556,204631


## 3. Agregación a nivel mes–localidad

En esta sección llevamos cada fuente a un grano común:

- **IBOCA** → promedio mensual por localidad (`iboca_mes_loc`)
- **SISAIRE** → promedio mensual por localidad (`sisaire_mes_loc`)
- **Medicina** → total mensual por localidad (`medicina_mes_loc`)

Todas estas tablas tendrán al menos: `Ano`, `Mes`, `Localidad`, `Fecha` (primer día del mes).


In [59]:
# 3.1 IBOCA → iboca_mes_loc (promedio mes–localidad)

def agregar_iboca_mensual(iboca_clean: pd.DataFrame) -> pd.DataFrame:
    if iboca_clean.empty:
        print("iboca_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame(columns=[
            "Ano", "Mes", "Localidad", "Fecha",
            "Concentracion_Promedio", "NowCast_Promedio", "IBOCA_Promedio"
        ])

    df = (
        iboca_clean
        .groupby(["Ano", "Mes", "Localidad"], as_index=False)
        .agg({
            "Concentracion": "mean",
            "NowCast": "mean",
            "IBOCA": "mean"
        })
    )

    # Fecha = primer día del mes correspondiente
    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    df.rename(
        columns={
            "Concentracion": "Concentracion_Promedio",
            "NowCast": "NowCast_Promedio",
            "IBOCA": "IBOCA_Promedio",
        },
        inplace=True
    )

    # Reordenar columnas
    cols = ["Ano", "Mes", "Localidad", "Fecha",
            "Concentracion_Promedio", "NowCast_Promedio", "IBOCA_Promedio"]
    df = df[cols]

    print("iboca_mes_loc listo. Shape:", df.shape)
    return df


iboca_mes_loc = agregar_iboca_mensual(iboca_clean)
iboca_mes_loc.head()


iboca_mes_loc listo. Shape: (1202, 7)


,Ano,Mes,Localidad,Fecha,Concentracion_Promedio,NowCast_Promedio,IBOCA_Promedio
0,2020.0,1.0,Carvajal - Sevillana,2020-01-01,29.820690,29.539651,87.655780
1,2020.0,1.0,Cdar,2020-01-01,13.808123,13.576747,49.557460
2,2020.0,1.0,Fontibon,2020-01-01,19.038043,19.359677,65.257769
3,2020.0,1.0,Guaymaral,2020-01-01,15.274105,14.988441,54.107608
4,2020.0,1.0,Kennedy,2020-01-01,23.707713,23.670296,74.957742


In [60]:
# 3.2 SISAIRE → sisaire_mes_loc (promedio mes–localidad)

def agregar_sisaire_mensual(sisaire_clean: pd.DataFrame) -> pd.DataFrame:
    if sisaire_clean.empty:
        print(" sisaire_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame(columns=[
            "Ano", "Mes", "Localidad", "Fecha",
            "NO2_Promedio", "O3_Promedio", "SO2_Promedio", "CO_Promedio"
        ])

    contaminantes = [c for c in ["NO2", "O3", "SO2", "CO"] if c in sisaire_clean.columns]

    df = (
        sisaire_clean
        .groupby(["Ano", "Mes", "Localidad"], as_index=False)
        .agg({c: "mean" for c in contaminantes})
    )

    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    # Renombrar a *_Promedio
    rename_dict = {c: f"{c}_Promedio" for c in contaminantes}
    df.rename(columns=rename_dict, inplace=True)

    cols = ["Ano", "Mes", "Localidad", "Fecha"] + list(rename_dict.values())
    df = df[cols]

    print("sisaire_mes_loc listo. Shape:", df.shape)
    return df


sisaire_mes_loc = agregar_sisaire_mensual(sisaire_clean)
sisaire_mes_loc.head()


sisaire_mes_loc listo. Shape: (1084, 8)


,Ano,Mes,Localidad,Fecha,NO2_Promedio,O3_Promedio,SO2_Promedio,CO_Promedio
0,2020,1,Carvajal - Sevillana,2020-01-01,44.197494,21.311795,8.618221,1110.075958
1,2020,1,Centro De Alto Rendimiento,2020-01-01,26.208586,34.033379,3.198179,912.327544
2,2020,1,Guaymaral,2020-01-01,21.656340,26.511976,NaN,NaN
3,2020,1,Kennedy,2020-01-01,33.802356,40.905034,4.382786,780.147598
4,2020,1,Las Ferias,2020-01-01,30.938985,22.892488,NaN,767.146783


In [61]:
# 3.3 MEDICINA → medicina_mes_loc (total mes–localidad)

def agregar_medicina_mensual(medicina_clean: pd.DataFrame) -> pd.DataFrame:
    if medicina_clean.empty:
        print("⚠️ medicina_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame()

    cols_muertes = [
        c for c in medicina_clean.columns
        if c.startswith("Muertes_")
    ]

    agg_dict = {c: "sum" for c in cols_muertes}

    df = (
        medicina_clean
        .groupby(["Ano", "Mes", "Localidad", "Fecha"], as_index=False)
        .agg(agg_dict)
    )

    print("medicina_mes_loc listo. Shape:", df.shape)
    return df


medicina_mes_loc = agregar_medicina_mensual(medicina_clean)
medicina_mes_loc.head()


medicina_mes_loc listo. Shape: (2482, 9)


,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,2015,1,00 - Bogotá,2015-01-01,188,112,76,188,188
1,2015,1,Antonio Narino,2015-01-01,2,1,1,2,2
2,2015,1,Barrios Unidos,2015-01-01,6,3,3,6,6
3,2015,1,Bolivia,2015-01-01,16,7,9,16,16
4,2015,1,Bosa,2015-01-01,20,14,6,20,20


In [62]:
# 3.4 POBLACIÓN → poblacion_mes_loc (total mes–localidad)

def agregar_poblacion_mensual(poblacion_clean: pd.DataFrame) -> pd.DataFrame:
    """
    Expande la población anual a nivel mes–localidad:
    - Para cada fila de poblacion_clean (Ano, Localidad, Poblacion_*)
      genera 12 meses (1..12).
    - Crea una columna Fecha = primer día del mes.
    """
    if poblacion_clean.empty:
        print("⚠️ poblacion_clean está vacío. Devuelvo un DataFrame vacío.")
        return pd.DataFrame(columns=[
            "Ano", "Mes", "Localidad", "Fecha",
            "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
        ])

    # DataFrame de meses (1 a 12)
    meses = pd.DataFrame({"Mes": range(1, 13)})

    # Producto cartesiano: cada fila de población × cada mes
    df = poblacion_clean.merge(meses, how="cross")

    # Fecha = primer día del mes correspondiente
    df["Fecha"] = pd.to_datetime(
        dict(year=df["Ano"], month=df["Mes"], day=1),
        errors="coerce"
    )

    cols = [
        "Ano", "Mes", "Localidad", "Fecha",
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
    ]
    df = df[cols].copy()

    print("poblacion_mes_loc listo. Shape:", df.shape)
    print("Ejemplo de localidades en poblacion_mes_loc:", df["Localidad"].unique()[:10])
    return df


poblacion_mes_loc = agregar_poblacion_mensual(poblacion_clean)
poblacion_mes_loc.head()


poblacion_mes_loc listo. Shape: (7812, 7)
Ejemplo de localidades en poblacion_mes_loc: ['Bogotá' 'Usaquen' 'Chapinero' 'Santa Fe' 'San Cristobal' 'Usme'
 'Tunjuelito' 'Bosa' 'Kennedy' 'Fontibon']


,Ano,Mes,Localidad,Fecha,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres
0,2005,1,Bogotá,2005-01-01,6710910,3236477,3474433
1,2005,2,Bogotá,2005-02-01,6710910,3236477,3474433
2,2005,3,Bogotá,2005-03-01,6710910,3236477,3474433
3,2005,4,Bogotá,2005-04-01,6710910,3236477,3474433
4,2005,5,Bogotá,2005-05-01,6710910,3236477,3474433


## 4. Construcción de dimensiones

A partir de las tablas mensuales por localidad, vamos a construir:

1. `dim_fecha`       → catálogo de meses (PK_Fecha, Año, Mes, NombreMes)
2. `dim_localidad`   → catálogo de localidades (PK_Localidad, Localidad, Codigo_Localidad)
3. `dim_localidad_fecha` → combinación Localidad + Fecha con población total, hombres y mujeres


In [63]:
# 4.1 DIM_FECHA  (pares (Ano,Mes) válidos = presentes en las 4 bases)

def construir_dim_fecha(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    medicina_mes_loc: pd.DataFrame,
    poblacion_mes_loc: pd.DataFrame,
):
    """
    Construye dim_fecha y calcula los AÑOS/MES VÁLIDOS como:
    - Pares (Ano, Mes) donde IBOCA tiene algún valor no nulo en sus columnas de promedio.
    - Pares (Ano, Mes) donde SISAIRE tiene algún contaminante promedio no nulo.
    - Pares (Ano, Mes) donde MEDICINA tiene Muertes_Totales > 0.
    - Pares (Ano, Mes) presentes en POBLACIÓN_MES (expansión mensual de población).

    Solo esos (Ano, Mes) se usan para dim_fecha.
    """

    # --- 1. Pares (Ano,Mes) con datos útiles en cada base mensual ---

    # IBOCA
    pares_iboca = set()
    if not iboca_mes_loc.empty:
        cols_ibo = [c for c in ["Concentracion_Promedio",
                                "NowCast_Promedio",
                                "IBOCA_Promedio"] if c in iboca_mes_loc.columns]
        if cols_ibo:
            mask_ibo = iboca_mes_loc[cols_ibo].notna().any(axis=1)
            tmp = iboca_mes_loc.loc[mask_ibo, ["Ano", "Mes"]].dropna()
            pares_iboca = set((int(a), int(m)) for a, m in zip(tmp["Ano"], tmp["Mes"]))

    # SISAIRE
    pares_sis = set()
    if not sisaire_mes_loc.empty:
        cols_sis = [c for c in ["NO2_Promedio",
                                "O3_Promedio",
                                "SO2_Promedio",
                                "CO_Promedio"] if c in sisaire_mes_loc.columns]
        if cols_sis:
            mask_sis = sisaire_mes_loc[cols_sis].notna().any(axis=1)
            tmp = sisaire_mes_loc.loc[mask_sis, ["Ano", "Mes"]].dropna()
            pares_sis = set((int(a), int(m)) for a, m in zip(tmp["Ano"], tmp["Mes"]))

    # MEDICINA
    pares_med = set()
    if (not medicina_mes_loc.empty
        and "Muertes_Totales" in medicina_mes_loc.columns
        and {"Ano", "Mes"}.issubset(medicina_mes_loc.columns)):
        mask_med = medicina_mes_loc["Muertes_Totales"] > 0
        tmp = medicina_mes_loc.loc[mask_med, ["Ano", "Mes"]].dropna()
        pares_med = set((int(a), int(m)) for a, m in zip(tmp["Ano"], tmp["Mes"]))

    # POBLACIÓN (ya está en formato mes–localidad)
    pares_pob = set()
    if not poblacion_mes_loc.empty and {"Ano", "Mes"}.issubset(poblacion_mes_loc.columns):
        tmp = poblacion_mes_loc[["Ano", "Mes"]].dropna().drop_duplicates()
        pares_pob = set((int(a), int(m)) for a, m in zip(tmp["Ano"], tmp["Mes"]))

    sets_pares = [pares_iboca, pares_sis, pares_med, pares_pob]
    sets_pares = [s for s in sets_pares if s]  # quitar vacíos

    if len(sets_pares) < 2:
        print("⚠️ No hay suficientes fuentes para cruzar años/meses.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), []

    # --- 2. Intersección de (Ano,Mes) con datos reales en TODAS las bases ---
    pares_validos = set.intersection(*sets_pares)
    pares_validos = sorted(pares_validos)
    print("Pares (Ano,Mes) presentes con datos REALES en TODAS las bases:", pares_validos)

    if not pares_validos:
        print("La intersección de (Ano,Mes) está vacía.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), []

    # Años válidos (derivados de los pares válidos)
    anos_validos = sorted({a for (a, _) in pares_validos})

    # --- 3. Construir dim_fecha solo con esos (Ano,Mes) ---

    frames = []
    for df in [iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc, poblacion_mes_loc]:
        if not df.empty and {"Ano", "Mes"}.issubset(df.columns):
            frames.append(df[["Ano", "Mes"]])

    if not frames:
        print("No hay datos mensuales para construir dim_fecha.")
        return pd.DataFrame(columns=["PK_Fecha", "Ano", "Mes", "NombreMes"]), anos_validos

    fechas = (
        pd.concat(frames, ignore_index=True)
        .dropna()
        .drop_duplicates()
    )

    fechas["Ano"] = fechas["Ano"].astype(int)
    fechas["Mes"] = fechas["Mes"].astype(int)
    fechas["__key__"] = list(zip(fechas["Ano"], fechas["Mes"]))

    pares_validos_set = set(pares_validos)
    fechas = fechas[fechas["__key__"].isin(pares_validos_set)].copy()
    fechas.drop(columns="__key__", inplace=True)

    fechas["PK_Fecha"] = fechas["Ano"] * 100 + fechas["Mes"]
    fechas["NombreMes"] = fechas["Mes"].map(MAPA_NOMBRE_MES)

    fechas = fechas.sort_values(["Ano", "Mes"]).reset_index(drop=True)
    fechas = fechas[["PK_Fecha", "Ano", "Mes", "NombreMes"]]

    print("dim_fecha lista. Shape:", fechas.shape)
    return fechas, anos_validos


# Ejecutar con las 4 tablas mensuales
dim_fecha, ANOS_VALIDOS = construir_dim_fecha(
    iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc, poblacion_mes_loc
)

dim_fecha.head()


Pares (Ano,Mes) presentes con datos REALES en TODAS las bases: [(2020, 1), (2020, 2), (2020, 3), (2020, 4), (2020, 5), (2020, 6), (2020, 7), (2020, 8), (2020, 9), (2020, 10), (2020, 11), (2020, 12), (2021, 1), (2021, 2), (2021, 3), (2021, 4), (2021, 5), (2021, 6), (2021, 7), (2021, 8), (2021, 9), (2021, 10), (2021, 11), (2021, 12), (2022, 1), (2022, 2), (2022, 3), (2022, 4), (2022, 5), (2022, 6), (2022, 7), (2022, 8), (2022, 9), (2022, 10), (2022, 11), (2022, 12), (2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12), (2025, 1), (2025, 2)]
dim_fecha lista. Shape: (62, 4)


,PK_Fecha,Ano,Mes,NombreMes
0,202001,2020,1,Enero
1,202002,2020,2,Febrero
2,202003,2020,3,Marzo
3,202004,2020,4,Abril
4,202005,2020,5,Mayo


In [64]:
# 4.2 DIM_LOCALIDAD (solo localidades con DATOS en TODAS las bases)

def construir_dim_localidad(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    medicina_mes_loc: pd.DataFrame,
    poblacion_clean: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye dim_localidad solo con las localidades que tienen
    DATOS ÚTILES en TODAS las bases:
      - IBOCA: alguna vez con Concentracion/NowCast/IBOCA no nulos
      - SISAIRE: alguna vez con al menos un contaminante (NO2/O3/SO2/CO) no nulo
      - MEDICINA: alguna vez con Muertes_Totales > 0
      - POBLACIÓN: aparece en poblacion_clean
    """

    # --- IBOCA: localidades con algún valor válido ---
    locs_iboca = set()
    if not iboca_mes_loc.empty and "Localidad" in iboca_mes_loc.columns:
        cols_ibo = [c for c in ["Concentracion_Promedio",
                                "NowCast_Promedio",
                                "IBOCA_Promedio"] if c in iboca_mes_loc.columns]
        if cols_ibo:
            mask_ibo = iboca_mes_loc[cols_ibo].notna().any(axis=1)
            locs_iboca = set(iboca_mes_loc.loc[mask_ibo, "Localidad"].dropna().unique())

    # --- SISAIRE: localidades con algún contaminante válido ---
    locs_sis = set()
    if not sisaire_mes_loc.empty and "Localidad" in sisaire_mes_loc.columns:
        cols_sis = [c for c in ["NO2_Promedio",
                                "O3_Promedio",
                                "SO2_Promedio",
                                "CO_Promedio"] if c in sisaire_mes_loc.columns]
        if cols_sis:
            mask_sis = sisaire_mes_loc[cols_sis].notna().any(axis=1)
            locs_sis = set(sisaire_mes_loc.loc[mask_sis, "Localidad"].dropna().unique())

    # --- MEDICINA: localidades con alguna muerte registrada ---
    locs_med = set()
    if (not medicina_mes_loc.empty
        and "Localidad" in medicina_mes_loc.columns
        and "Muertes_Totales" in medicina_mes_loc.columns):
        mask_med = medicina_mes_loc["Muertes_Totales"] > 0
        locs_med = set(medicina_mes_loc.loc[mask_med, "Localidad"].dropna().unique())

    # --- POBLACIÓN: localidades presentes en población_clean ---
    locs_pob = set()
    if not poblacion_clean.empty and "Localidad" in poblacion_clean.columns:
        locs_pob = set(poblacion_clean["Localidad"].dropna().unique())

    sets_locs = [locs_iboca, locs_sis, locs_med, locs_pob]
    sets_locs = [s for s in sets_locs if s]  # quitar sets vacíos

    if len(sets_locs) < 2:
        print("⚠️ No hay suficientes fuentes para cruzar localidades.")
        return pd.DataFrame(columns=["PK_Localidad", "Localidad", "Codigo_Localidad"])

    # Intersección de localidades con datos reales en todas las bases
    locs_intersection = set.intersection(*sets_locs)
    print("Número de localidades con DATOS en todas las bases:", len(locs_intersection))

    if not locs_intersection:
        print("⚠️ La intersección de localidades está vacía.")
        return pd.DataFrame(columns=["PK_Localidad", "Localidad", "Codigo_Localidad"])

    base_locs = pd.DataFrame({"Localidad": sorted(locs_intersection)})

    # Traer Código de localidad desde población
    locs_pob_df = (
        poblacion_clean[["Codigo_Localidad", "Localidad"]]
        .drop_duplicates(subset=["Localidad"])
    )

    dim_loc = base_locs.merge(locs_pob_df, on="Localidad", how="left")

    # asegurar tipo numérico del código (por si lo quieres para joins)
    dim_loc["Codigo_Localidad"] = pd.to_numeric(dim_loc["Codigo_Localidad"], errors="coerce")

    # Crear PK_Localidad
    dim_loc = dim_loc.sort_values("Localidad").reset_index(drop=True)
    dim_loc.insert(0, "PK_Localidad", dim_loc.index + 1)

    print("dim_localidad lista. Shape:", dim_loc.shape)
    return dim_loc


dim_localidad = construir_dim_localidad(
    iboca_mes_loc, sisaire_mes_loc, medicina_mes_loc, poblacion_clean
)
dim_localidad.head()


Número de localidades con DATOS en todas las bases: 8
dim_localidad lista. Shape: (8, 3)


,PK_Localidad,Localidad,Codigo_Localidad
0,1,Bolivia,19
1,2,Fontibon,9
2,3,Kennedy,8
3,4,Puente Aranda,16
4,5,San Cristobal,4


In [65]:
# 4.3 DIM_LOCALIDAD_FECHA (redefinida por si acaso)

def construir_dim_localidad_fecha(
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
    poblacion_clean: pd.DataFrame,
) -> pd.DataFrame:
    if dim_fecha.empty or dim_localidad.empty:
        print("⚠️ dim_fecha o dim_localidad están vacías. Devuelvo DataFrame vacío.")
        return pd.DataFrame()

    # Base: todas las combinaciones de Fecha (PK_Fecha, Ano, Mes) x Localidad
    base = (
        dim_fecha[["PK_Fecha", "Ano", "Mes"]]
        .assign(key=1)
        .merge(
            dim_localidad[["PK_Localidad", "Localidad"]].assign(key=1),
            on="key", how="outer"
        )
        .drop(columns="key")
    )

    # Unimos con población por Año y Localidad
    # poblacion_clean: Ano, Codigo_Localidad, Localidad, Poblacion_*
    pob = poblacion_clean.copy()

    df = base.merge(
        pob,
        on=["Ano", "Localidad"],
        how="left",
        suffixes=("", "_pob")
    )

    # Crear PK_LocalidadFecha
    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_LocalidadFecha", df.index + 1)

    columnas = [
        "PK_LocalidadFecha",
        "PK_Fecha", "PK_Localidad",
        "Ano", "Mes", "Localidad",
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres"
    ]
    # Por si alguna columna de población no existe:
    columnas = [c for c in columnas if c in df.columns]

    df = df[columnas]

    print("dim_localidad_fecha lista. Shape:", df.shape)
    return df

# 👇 MUY IMPORTANTE: crear realmente la tabla
dim_localidad_fecha = construir_dim_localidad_fecha(
    dim_fecha, dim_localidad, poblacion_clean
)

dim_localidad_fecha.head()


dim_localidad_fecha lista. Shape: (496, 9)


,PK_LocalidadFecha,PK_Fecha,PK_Localidad,Ano,Mes,Localidad,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres
0,1,202001,1,2020,1,Bolivia,642989,315843,327146
1,2,202001,2,2020,1,Fontibon,383577,181305,202272
2,3,202001,3,2020,1,Kennedy,1046951,503223,543728
3,4,202001,4,2020,1,Puente Aranda,250855,122500,128355
4,5,202001,5,2020,1,San Cristobal,399766,193584,206182


## 5. Construcción de tablas de hechos

Con las dimensiones ya listas, construimos:

- `fact_mortalidad` → mortalidad mensual por localidad.
- `fact_momento`    → contaminación mensual por localidad (IBOCA + SISAIRE).

Ambas referencian `dim_fecha` y `dim_localidad` a través de sus PKs.


In [66]:
# 5.1 FACT_MORTALIDAD (solo localidades presentes en dim_localidad)

def construir_fact_mortalidad(
    medicina_mes_loc: pd.DataFrame,
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
) -> pd.DataFrame:
    if medicina_mes_loc.empty:
        print("medicina_mes_loc está vacío. Devuelvo DataFrame vacío.")
        return pd.DataFrame()

    df = medicina_mes_loc.merge(
        dim_fecha[["PK_Fecha", "Ano", "Mes"]],
        on=["Ano", "Mes"],
        how="left"
    )

    df = df.merge(
        dim_localidad[["PK_Localidad", "Localidad"]],
        on="Localidad",
        how="left"
    )

    # Eliminar filas cuya localidad no esté en la intersección (PK_Localidad nulo)
    df = df.dropna(subset=["PK_Localidad"])

    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_Mortalidad", df.index + 1)

    cols_muertes = [c for c in df.columns if c.startswith("Muertes_")]

    columnas = (
        ["PK_Mortalidad", "PK_Fecha", "PK_Localidad", "Ano", "Mes", "Localidad", "Fecha"]
        + cols_muertes
    )
    fact = df[columnas]

    print("fact_mortalidad lista. Shape:", fact.shape)
    return fact


fact_mortalidad = construir_fact_mortalidad(
    medicina_mes_loc, dim_fecha, dim_localidad
)
fact_mortalidad.head()


fact_mortalidad lista. Shape: (975, 12)


,PK_Mortalidad,PK_Fecha,PK_Localidad,Ano,Mes,Localidad,Fecha,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Muertes_CIE10_Agrupada,Muertes_CIE10_Basica
0,1,NaN,1.0,2015,1,Bolivia,2015-01-01,16,7,9,16,16
1,2,NaN,2.0,2015,1,Fontibon,2015-01-01,12,7,5,12,12
2,3,NaN,3.0,2015,1,Kennedy,2015-01-01,28,15,13,28,28
3,4,NaN,4.0,2015,1,Puente Aranda,2015-01-01,5,4,1,5,5
4,5,NaN,5.0,2015,1,San Cristobal,2015-01-01,16,9,7,16,16


In [67]:
# 5.2 FACT_MOMENTO (solo localidades presentes en dim_localidad)

def construir_fact_momento(
    iboca_mes_loc: pd.DataFrame,
    sisaire_mes_loc: pd.DataFrame,
    dim_fecha: pd.DataFrame,
    dim_localidad: pd.DataFrame,
    fact_mortalidad: pd.DataFrame,
) -> pd.DataFrame:
    if iboca_mes_loc.empty and sisaire_mes_loc.empty:
        print("⚠️ No hay datos de iboca_mes_loc ni sisaire_mes_loc.")
        return pd.DataFrame()

    df = pd.merge(
        iboca_mes_loc,
        sisaire_mes_loc,
        on=["Ano", "Mes", "Localidad", "Fecha"],
        how="outer"
    )

    df = df.merge(
        dim_fecha[["PK_Fecha", "Ano", "Mes"]],
        on=["Ano", "Mes"],
        how="left"
    )

    df = df.merge(
        dim_localidad[["PK_Localidad", "Localidad"]],
        on="Localidad",
        how="left"
    )

    # Filtrar a localidades presentes en dim_localidad
    df = df.dropna(subset=["PK_Localidad"])

    if not fact_mortalidad.empty:
        df = df.merge(
            fact_mortalidad[["PK_Mortalidad", "PK_Fecha", "PK_Localidad"]],
            on=["PK_Fecha", "PK_Localidad"],
            how="left"
        )
        df.rename(columns={"PK_Mortalidad": "FK_Mortalidad"}, inplace=True)
    else:
        df["FK_Mortalidad"] = None

    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)
    df.insert(0, "PK_Momento", df.index + 1)

    columnas = [
        "PK_Momento",
        "PK_Fecha",
        "PK_Localidad",
        "FK_Mortalidad",
        "Ano", "Mes", "Localidad", "Fecha",
        "Concentracion_Promedio",
        "NowCast_Promedio",
        "IBOCA_Promedio",
        "NO2_Promedio",
        "O3_Promedio",
        "SO2_Promedio",
        "CO_Promedio",
    ]
    columnas_finales = [c for c in columnas if c in df.columns]

    fact = df[columnas_finales]

    print("fact_momento lista. Shape:", fact.shape)
    return fact


fact_momento = construir_fact_momento(
    iboca_mes_loc,
    sisaire_mes_loc,
    dim_fecha,
    dim_localidad,
    fact_mortalidad,
)
fact_momento.head()


fact_momento lista. Shape: (3831, 15)


,PK_Momento,PK_Fecha,PK_Localidad,FK_Mortalidad,Ano,Mes,Localidad,Fecha,Concentracion_Promedio,NowCast_Promedio,IBOCA_Promedio,NO2_Promedio,O3_Promedio,SO2_Promedio,CO_Promedio
0,1,202001.0,2.0,481,2020.0,1.0,Fontibon,2020-01-01,19.038043,19.359677,65.257769,NaN,NaN,NaN,NaN
1,2,202001.0,3.0,482,2020.0,1.0,Kennedy,2020-01-01,23.707713,23.670296,74.957742,33.802356,40.905034,4.382786,780.147598
2,3,202001.0,4.0,483,2020.0,1.0,Puente Aranda,2020-01-01,14.323288,14.688498,52.628796,31.752486,19.387378,4.396192,541.464885
3,4,202001.0,5.0,484,2020.0,1.0,San Cristobal,2020-01-01,11.977870,12.309655,45.722772,NaN,26.296755,NaN,NaN
4,5,202001.0,6.0,485,2020.0,1.0,Suba,2020-01-01,16.752368,16.484885,58.322901,NaN,28.271755,5.939822,NaN


In [68]:
# 5.3 FACT_TASAS_MORTALIDAD (mismo estilo que fact_momento)

def construir_fact_tasas_mortalidad(
    dim_localidad_fecha: pd.DataFrame,
    fact_mortalidad: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye la tabla puente fact_tasas_mortalidad con el mismo formato
    y patrón que fact_momento:
    
    - Llaves PK_Fecha y PK_Localidad
    - FK_Mortalidad
    - PK_Tasa autoincremental
    - Ordenado y filtrado
    """

    if dim_localidad_fecha.empty or fact_mortalidad.empty:
        print("No hay datos para construir fact_tasas_mortalidad.")
        return pd.DataFrame()
    
    # 1) MERGE base: población y mortalidad
    df = dim_localidad_fecha.merge(
        fact_mortalidad[[
            "PK_Mortalidad",
            "PK_Fecha",
            "PK_Localidad",
            "Muertes_Totales",
            "Muertes_Totales_Hombres",
            "Muertes_Totales_Mujeres"
        ]],
        on=["PK_Fecha", "PK_Localidad"],
        how="inner"   # solo cruces válidos
    )

    # Renombrar llave foránea
    df.rename(columns={"PK_Mortalidad": "FK_Mortalidad"}, inplace=True)

    # Asegurar numéricos básicos (por si acaso)
    for c in [
        "Poblacion_Total", "Poblacion_Hombres", "Poblacion_Mujeres",
        "Muertes_Totales", "Muertes_Totales_Hombres", "Muertes_Totales_Mujeres"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Evitar división por cero: donde población = 0, ponemos NaN
    df.loc[df["Poblacion_Total"]   == 0, "Poblacion_Total"]   = pd.NA
    df.loc[df["Poblacion_Hombres"] == 0, "Poblacion_Hombres"] = pd.NA
    df.loc[df["Poblacion_Mujeres"] == 0, "Poblacion_Mujeres"] = pd.NA

    # 2) Calcular tasas
    df["Tasa_Mortalidad_Total"]   = df["Muertes_Totales"]          / df["Poblacion_Total"]
    df["Tasa_Mortalidad_Hombres"] = df["Muertes_Totales_Hombres"]  / df["Poblacion_Hombres"]
    df["Tasa_Mortalidad_Mujeres"] = df["Muertes_Totales_Mujeres"]  / df["Poblacion_Mujeres"]

    # Tasas por 100.000
    factor = 100000
    df["Tasa_Total_100k"]   = df["Tasa_Mortalidad_Total"]   * factor
    df["Tasa_Hombres_100k"] = df["Tasa_Mortalidad_Hombres"] * factor
    df["Tasa_Mujeres_100k"] = df["Tasa_Mortalidad_Mujeres"] * factor

    # 3) Ordenar
    df = df.sort_values(["Ano", "Mes", "Localidad"]).reset_index(drop=True)

    # 4) Crear PK autoincremental
    df.insert(0, "PK_Tasa", df.index + 1)

    # 5) Orden de columnas como fact_momento
    columnas = [
        "PK_Tasa",
        "PK_Fecha",
        "PK_Localidad",
        "FK_Mortalidad",
        "Ano",
        "Mes",
        "Localidad",
        "Fecha",
        "Poblacion_Total",
        "Poblacion_Hombres",
        "Poblacion_Mujeres",
        "Muertes_Totales",
        "Muertes_Totales_Hombres",
        "Muertes_Totales_Mujeres",
        "Tasa_Mortalidad_Total",
        "Tasa_Mortalidad_Hombres",
        "Tasa_Mortalidad_Mujeres",
        "Tasa_Total_100k",
        "Tasa_Hombres_100k",
        "Tasa_Mujeres_100k",
    ]

    columnas_finales = [c for c in columnas if c in df.columns]
    fact_tasas = df[columnas_finales]

    print("fact_tasas_mortalidad lista. Shape:", fact_tasas.shape)
    return fact_tasas


# Ejecutar
fact_tasas_mortalidad = construir_fact_tasas_mortalidad(
    dim_localidad_fecha,
    fact_mortalidad
)

fact_tasas_mortalidad.head()


fact_tasas_mortalidad lista. Shape: (496, 19)


,PK_Tasa,PK_Fecha,PK_Localidad,FK_Mortalidad,Ano,Mes,Localidad,Poblacion_Total,Poblacion_Hombres,Poblacion_Mujeres,Muertes_Totales,Muertes_Totales_Hombres,Muertes_Totales_Mujeres,Tasa_Mortalidad_Total,Tasa_Mortalidad_Hombres,Tasa_Mortalidad_Mujeres,Tasa_Total_100k,Tasa_Hombres_100k,Tasa_Mujeres_100k
0,1,202001,1,480,2020,1,Bolivia,642989.0,315843.0,327146.0,16,14,2,0.000025,0.000044,0.000006,2.488378,4.432582,0.611348
1,2,202001,2,481,2020,1,Fontibon,383577.0,181305.0,202272.0,4,2,2,0.000010,0.000011,0.000010,1.042815,1.103114,0.988768
2,3,202001,3,482,2020,1,Kennedy,1046951.0,503223.0,543728.0,23,11,12,0.000022,0.000022,0.000022,2.196855,2.185910,2.206986
3,4,202001,4,483,2020,1,Puente Aranda,250855.0,122500.0,128355.0,9,2,7,0.000036,0.000016,0.000055,3.587730,1.632653,5.453625
4,5,202001,5,484,2020,1,San Cristobal,399766.0,193584.0,206182.0,11,5,6,0.000028,0.000026,0.000029,2.751610,2.582858,2.910050


## 6. Exportar tablas finales a CSV

Por último, guardamos **solo** las 6 tablas del modelo final:

1. `dim_fecha.csv`
2. `dim_localidad.csv`
3. `dim_localidad_fecha.csv`
4. `fact_mortalidad.csv`
5. `fact_momento.csv`
6. `fact_tasas_mortalidad.csv`


In [ ]:
# ============================================================
# 💾 EXPORTAR TABLAS DEFINITIVAS FILTRADAS
# ============================================================

dim_fecha.to_csv(RUTA_SALIDA / "dim_fecha.csv", index=False)
dim_localidad.to_csv(RUTA_SALIDA / "dim_localidad.csv", index=False)
dim_localidad_fecha.to_csv(RUTA_SALIDA / "dim_localidad_fecha.csv", index=False)
fact_mortalidad.to_csv(RUTA_SALIDA / "fact_mortalidad.csv", index=False)
fact_momento.to_csv(RUTA_SALIDA / "fact_momento.csv", index=False)
fact_tasas_mortalidad.to_csv(RUTA_SALIDA / "fact_tasas_mortalidad.csv", index=False)
